# 🛡️ SafeAd AI (SAFE-VISION) - Google Colab Setup Notebook
This notebook verifies GPU hardware availability, installs dependencies, mounts Google Drive, downloads genuine datasets (Option B & Option C), and executes a dry-run advertisement safety test on Google Colab Free.

In [ ]:
# Step 1: Check GPU & Hardware Environment
import torch
import sys
import os

print("==========================================================")
print("            GOOGLE COLAB ENVIRONMENT MONITOR               ")
print("==========================================================")
print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name       : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**2)
    print(f"Total VRAM     : {total_vram:.2f} MB")
print("==========================================================")

In [ ]:
# Step 2: Mount Google Drive for Persistent Model/Dataset Storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = '/content/drive/MyDrive/SafeAdAI'
    os.makedirs(os.path.join(drive_path, 'datasets'), exist_ok=True)
    print(f"[SafeAd AI] Google Drive mounted successfully at: {drive_path}")
except ImportError:
    print("[SafeAd AI] Not running inside Google Colab interactive environment. Skipping Drive mount.")

In [ ]:
# Step 3: Install Required Dependencies & HuggingFace Datasets Library
!pip install -q opencv-python pytesseract faiss-cpu scikit-learn pillow pydantic requests pyjwt passlib datasets kaggle

In [ ]:
# Step 4 [OPTION B]: Stream Genuine Academic Datasets via HuggingFace Hub
from datasets import load_dataset

print("==========================================================")
print("    OPTION B: HUGGINGFACE DATASET STREAMING (GENUINE)     ")
print("==========================================================")
try:
    # Stream genuine UCF-Crime video dataset
    ucf_stream = load_dataset("jinmang2/ucf_crime", streaming=True)
    print("✓ Connected to genuine UCF-Crime dataset on HuggingFace Hub!")
    first_sample = next(iter(ucf_stream["train"]))
    print(f"Sample attributes retrieved: {list(first_sample.keys())}")
except Exception as e:
    print(f"HuggingFace Streaming Note: {e}")

In [ ]:
# Step 5 [OPTION C]: Download Genuine Datasets via Kaggle API to Google Drive
print("==========================================================")
print("    OPTION C: KAGGLE GENUINE DATASET DOWNLOAD TO DRIVE    ")
print("==========================================================")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")

if not os.path.exists(kaggle_json_path):
    print("INFO: Upload your 'kaggle.json' API key file to Colab root to run automated full dataset downloads.")
else:
    print("✓ Kaggle API credentials verified.")
    !kaggle datasets download -d jessicali9530/utkface-new --path /content/drive/MyDrive/SafeAdAI/datasets/utkface/ --unzip
    !kaggle datasets download -d mateohervas/ucf-crime-full-dataset --path /content/drive/MyDrive/SafeAdAI/datasets/ucf_crime/ --unzip

In [ ]:
# Step 6: Verify Memory Management & Pipeline Execution
from ai.memory_manager import MemoryManager
from ai.pipeline import run_safead_inference

MemoryManager.print_resource_status()

import numpy as np
from PIL import Image
test_img = np.ones((400, 600, 3), dtype=np.uint8) * 240
test_path = "colab_test_ad.jpg"
Image.fromarray(test_img).save(test_path)

res = run_safead_inference(test_path, title="Grand Casino Offer", caption="Win real cash satta online")
print("\n[SafeAd AI Setup Dry-Run Output]:")
import json
print(json.dumps(res, indent=2))